In [1]:
import random

class AlgorithmeGenetique:
    """
    Un moteur générique d'algorithme génétique.
    Il manipule uniquement de l'ADN binaire. Le sens de cet ADN est défini par la fonction de fitness.
    """
    def __init__(self, fonction_fitness, taille_adn, taille_population=100, taux_mutation=0.05, max_generations=100):
        self.fonction_fitness = fonction_fitness  # La fonction qui évalue la "survie" (plus le score est haut, mieux c'est)
        self.taille_adn = taille_adn              # La longueur de la chaîne binaire
        self.taille_population = taille_population
        self.taux_mutation = taux_mutation
        self.max_generations = max_generations

    def _initialiser_population(self):
        """Crée la population de départ avec des séquences d'ADN aléatoires."""
        return ["".join(random.choice("01") for _ in range(self.taille_adn)) for _ in range(self.taille_population)]

    def _croisement(self, parent_a, parent_b):
        """Croisement à un point aléatoire pour créer deux enfants."""
        point_coupure = random.randint(1, self.taille_adn - 1)
        enfant_1 = parent_a[:point_coupure] + parent_b[point_coupure:]
        enfant_2 = parent_b[:point_coupure] + parent_a[point_coupure:]
        return enfant_1, enfant_2

    def _mutation(self, enfant):
        """Bascule aléatoirement certains bits selon le taux de mutation."""
        enfant_mute = ""
        for bit in enfant:
            if random.random() < self.taux_mutation:
                enfant_mute += "1" if bit == "0" else "0"
            else:
                enfant_mute += bit
        return enfant_mute

    def run(self):
        """Lance la boucle d'évolution de Darwin."""
        population = self._initialiser_population()
        meilleur_absolu = None
        meilleur_score_absolu = -1

        for generation in range(self.max_generations):
            # 1. Évaluation : On note chaque individu
            scores = [(individu, self.fonction_fitness(individu)) for individu in population]
            
            # On trie du meilleur score au pire
            scores.sort(key=lambda x: x[1], reverse=True)
            
            # Sauvegarde du meilleur de tous les temps
            if scores[0][1] > meilleur_score_absolu:
                meilleur_absolu = scores[0][0]
                meilleur_score_absolu = scores[0][1]

            # 2. Sélection : On garde la meilleure moitié (les "bons parents")
            meilleurs_parents = [ind[0] for ind in scores[:self.taille_population // 2]]
            
            # 3. Reproduction : On crée la nouvelle génération
            nouvelle_population = []
            while len(nouvelle_population) < self.taille_population:
                # On choisit deux parents au hasard parmi l'élite
                parent_a, parent_b = random.sample(meilleurs_parents, 2)
                
                # Croisement
                enfant_1, enfant_2 = self._croisement(parent_a, parent_b)
                
                # Mutation et ajout à la nouvelle population
                nouvelle_population.append(self._mutation(enfant_1))
                if len(nouvelle_population) < self.taille_population:
                    nouvelle_population.append(self._mutation(enfant_2))

            population = nouvelle_population

        return meilleur_absolu, meilleur_score_absolu

In [2]:
# --- DÉFINITION DE NOTRE PROBLÈME SPÉCIFIQUE ---

def decoder_adn_en_x(adn):
    """Transforme l'ADN binaire en un nombre entier (pour simplifier)."""
    # Convertit la chaîne binaire (ex: "0101") en entier base 2 (ex: 5)
    return int(adn, 2) 

def evaluer_equation(adn):
    """
    C'est NOTRE fonction de fitness. 
    L'algorithme veut toujours maximiser le score.
    Ici, si le résultat de l'équation est proche de 0, le score doit être très grand !
    """
    x = decoder_adn_en_x(adn)
    
    # Notre équation : x^2 - 5x + 6
    resultat = (x**2) - (5*x) + 6
    erreur = abs(resultat)
    
    # Astuce mathématique : On inverse l'erreur pour que 
    # si l'erreur = 0, le score soit immense (ou infini, on évite la div par 0)
    if erreur == 0:
        return 999999 # Score parfait !
    else:
        return 1.0 / erreur

# --- LANCEMENT DE L'ALGORITHME ---

# On a besoin d'un ADN qui peut représenter des chiffres. 
# 4 bits permettent d'aller de 0 à 15 (ex: "1111" = 15). C'est suffisant pour tester.
taille_adn = 4 

# On crée une instance de notre classe générique en lui passant NOTRE problème
mon_ia = AlgorithmeGenetique(
    fonction_fitness=evaluer_equation, 
    taille_adn=taille_adn, 
    taille_population=20, 
    taux_mutation=0.1, 
    max_generations=50
)

# On lance l'évolution !
meilleur_adn, score = mon_ia.run()

# On affiche les résultats
x_trouve = decoder_adn_en_x(meilleur_adn)
print(f"Meilleur ADN trouvé : {meilleur_adn}")
print(f"Ce qui correspond au nombre x = {x_trouve}")
print(f"Vérification de l'équation : {x_trouve}^2 - 5({x_trouve}) + 6 = {(x_trouve**2) - (5*x_trouve) + 6}")

Meilleur ADN trouvé : 0010
Ce qui correspond au nombre x = 2
Vérification de l'équation : 2^2 - 5(2) + 6 = 0
